In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

/home/foolmann/miniconda3/envs/genaienv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# tool create 
@tool
def multiply(a: int, b: int ) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a*b

In [6]:
print(multiply.invoke({'a':10,'b':10}))

100


In [7]:
multiply.name

'multiply'

In [8]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [9]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

## Tool Binding

In [10]:
# Tool binding 

In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.2)

bind_tools functions take the as many as tools as the list 

In [15]:
llm_with_tools = llm.bind_tools([multiply])

Now this is llm but with the access to the tools we created for multiplication 

## Tool Calling 

In [16]:
llm_with_tools.invoke("Hi how are you ?")

AIMessage(content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today?", 'extras': {'signature': 'EjQKMgERTTIPZxECuo2jBD5tZXVDAo+Yj6chV7TYe+6VbLhJps/IU1TY+yNQW4lH66YSVfU7'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fb2db-eb84-76c3-8e46-1aad0cf8d6b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 17, 'total_tokens': 82, 'input_token_details': {'cache_read': 0}})

This is simple response 

In [18]:
response = llm_with_tools.invoke("can you multiply 3 with 10 ? ")

##### Now here the llm looks at the prompt and thinks that tool usage is neccessary and decides to suggest the tool and argument with the list of the tool_calls instead of answering the query itself 

 tool_calls=[{'name': 'multiply', 'args': {'b': 10, 'a': 3}, 'id': '1aecb900-cbfd-49bd-adc8-521a8594483c', 'type': 'tool_call'}] 

In [20]:
response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'effff7e5-5df0-4222-91e1-dfb6d73ec68d': 'EjQKMgERTTIPfR1dO1+heLvu19N2PpaVwcWIrfK+BEyNp8rjVbNzQG4ziBWwkjfohQHgHVRR'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fb2dd-6a37-7d62-a5fb-eaf2a67b7818-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'effff7e5-5df0-4222-91e1-dfb6d73ec68d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 17, 'total_tokens': 88, 'input_token_details': {'cache_read': 0}})

Now this response . The content is empty so no actual text output is missing. But we can see the tool_calls attribute with the tools that llm thinks the best suited. It gives the list of the tool_calls which have content of the too name and the arguments as the dictionary/ json 

In [ ]:
response.tool_calls
# this is the list of tool calls so there can be multiple tools 

[{'name': 'multiply',
  'args': {'a': 3, 'b': 10},
  'id': 'effff7e5-5df0-4222-91e1-dfb6d73ec68d',
  'type': 'tool_call'}]

In [ ]:
# getting the first tool call ie first and only one in this case 
response.tool_calls[0]


{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'effff7e5-5df0-4222-91e1-dfb6d73ec68d',
 'type': 'tool_call'}

We didnot get the answer the llm only sugggested the argument and tool . This is to avoid the execution of the tools that can be harmful this gives control to the programmmer user to take the action of using or not using the tool 

## Tool Execution 

In [24]:
arguments = response.tool_calls[0]['args']
arguments

{'a': 3, 'b': 10}

In [25]:
multiply.invoke(arguments)

30

If we just pass the arguments suggested we will get simple result. 

In [26]:
multiply.invoke({'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'effff7e5-5df0-4222-91e1-dfb6d73ec68d',
 'type': 'tool_call'})

ToolMessage(content='30', name='multiply', tool_call_id='effff7e5-5df0-4222-91e1-dfb6d73ec68d')

In [ ]:
# OR 
multiply.invoke(response.tool_calls[0])

ToolMessage(content='30', name='multiply', tool_call_id='effff7e5-5df0-4222-91e1-dfb6d73ec68d')

If we send the entire tool_calls  ie arguments with all the metadata . We will get the organized wrapped TOOL MESSAGE . 

This is better as we can now send this message to the llm and llm will have the proper understanding of the who and how the result was produced . It understand it was generated by the tool 